# FMP Equities

Read-first examples for FMP-backed equity data in Quant Warehouse. Set `RUN_REFRESH = True` only when you want to download missing data.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

EQUITY_SYMBOL = "AAPL"

In [2]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

## Available Equity Sections

In [3]:
equity_sections = ["prices", "profile", *FMP_HISTORICAL_EQUITY_SECTIONS, *FMP_EXTENDED_EQUITY_SECTIONS]
equity_route_libraries = {
    "prices": EQUITY_PRICES_LIBRARY,
    "profile": "profile",
    **{section: fundamental_library(section) for section in FMP_HISTORICAL_EQUITY_SECTIONS},
    **{section: fundamental_library(section) for section in FMP_EXTENDED_EQUITY_SECTIONS},
}
route_table(equity_route_libraries)

,section,openbb_route,provider_library
0,prices,equity.price.historical,fmp_equity_prices
1,profile,equity.profile,fmp_profile
2,income,equity.fundamental.income,fmp_equity_fundamental_income
3,balance,equity.fundamental.balance,fmp_equity_fundamental_balance
4,cash,equity.fundamental.cash,fmp_equity_fundamental_cash
5,metrics,equity.fundamental.metrics,fmp_equity_fundamental_metrics
6,ratios,equity.fundamental.ratios,fmp_equity_fundamental_ratios
7,income_growth,equity.fundamental.income_growth,fmp_equity_fundamental_income_growth
8,balance_growth,equity.fundamental.balance_growth,fmp_equity_fundamental_balance_growth
9,cash_growth,equity.fundamental.cash_growth,fmp_equity_fundamental_cash_growth


## Local Storage Coverage

In [4]:
state_table(EQUITY_SYMBOL, equity_sections)

,symbol,section,provider,min_date,max_date,row_count,column_count,columns_present,last_fetched_at
0,AAPL,prices,fmp,1980-12-12,2026-06-24,11474,8,"(close, fmp__change, fmp__change_percent, fmp_...",2026-06-25T13:33:10.981691+00:00
1,AAPL,profile,fmp,None,None,1,20,"(beta, companyName, company_name, country, exc...",2026-06-25T20:26:47.302228+00:00
2,AAPL,income,fmp,1985-09-30,2026-05-01,162,32,"(accepted_date, bottom_line_net_income, cost_a...",2026-06-18T03:31:23.508677+00:00
3,AAPL,balance,fmp,1985-09-30,2026-05-01,150,54,"(accepted_date, account_payables, accounts_rec...",2026-06-18T03:33:10.289140+00:00
4,AAPL,cash,fmp,1989-12-31,2026-05-01,145,40,"(accepted_date, accounts_payables, accounts_re...",2026-06-18T03:35:17.994410+00:00
5,AAPL,metrics,fmp,1985-09-30,2026-06-25,165,46,"(average_inventory, average_payables, average_...",2026-06-25T13:35:12.405872+00:00
6,AAPL,ratios,fmp,1985-09-30,2026-06-25,165,71,"(asset_turnover, book_value_per_share, bottom_...",2026-06-25T13:35:13.104601+00:00
7,AAPL,income_growth,fmp,1985-09-30,2026-03-28,163,39,"(fiscal_period, growth_basic_earings_per_share...",2026-06-25T20:26:52.149165+00:00
8,AAPL,balance_growth,fmp,1985-09-30,2026-03-28,151,57,"(fiscal_period, growth_account_payables, growt...",2026-06-25T20:26:52.473059+00:00
9,AAPL,cash_growth,fmp,1989-12-31,2026-03-28,146,51,"(fiscal_period, growth_account_payable, growth...",2026-06-25T20:26:52.798278+00:00


## Prices and Profile

In [5]:
if RUN_REFRESH:
    warehouse.refresh_prices(EQUITY_SYMBOL, providers=[PROVIDER])
    warehouse.refresh_profile(EQUITY_SYMBOL, provider=PROVIDER)

preview(warehouse.read_prices(EQUITY_SYMBOL, provider=PROVIDER))

,open,high,low,close,volume,fmp__vwap,fmp__change,fmp__change_percent
date,,,,,,,,
2026-06-17,300.85,302.07,294.36,295.95,42745100,298.3075,-4.90,-0.016300
2026-06-18,298.11,300.57,295.62,298.01,85962201,298.0775,-0.10,-0.000335
2026-06-22,297.31,302.42,296.76,297.01,44879914,298.3750,-0.30,-0.001009
2026-06-23,297.54,301.64,294.18,294.30,52010929,296.9150,-3.24,-0.010900
2026-06-24,295.36,299.70,292.94,293.08,53081859,295.2700,-2.28,-0.007719


In [6]:
profile = warehouse.read_profile(EQUITY_SYMBOL, provider=PROVIDER)
asdict(profile) if profile is not None else None

{'symbol': 'AAPL',
 'provider': 'fmp',
 'source_provider': 'fmp_screener',
 'fetched_at': '2026-06-25T20:26:47.300708+00:00',
 'company_name': 'Apple Inc.',
 'exchange': 'NASDAQ',
 'country': 'US',
 'sector': 'Technology',
 'industry': 'Consumer Electronics',
 'market_cap': 4322488870800.0,
 'beta': 1.086,
 'cik': None,
 'ipo_date': None,
 'payload': {'symbol': 'AAPL',
  'companyName': 'Apple Inc.',
  'marketCap': 4322488870800.0,
  'sector': 'Technology',
  'industry': 'Consumer Electronics',
  'beta': 1.086,
  'price': 275.15,
  'lastAnnualDividend': 1.05,
  'volume': 102383867.0,
  'exchange': 'NASDAQ',
  'exchangeShortName': 'NASDAQ',
  'country': 'US',
  'isEtf': False,
  'isFund': False,
  'isActivelyTrading': True,
  'name': 'Apple Inc.',
  'market_cap': 4322488870800.0,
  'is_etf': False,
  'is_fund': False,
  'company_name': 'Apple Inc.'}}

## Core Fundamentals

In [7]:
if RUN_REFRESH:
    warehouse.refresh_fundamentals(
        EQUITY_SYMBOL,
        sections=["income", "balance", "cash", "metrics", "ratios"],
        providers=[PROVIDER],
        period="quarter",
    )

{
    "income": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="income", provider=PROVIDER)),
    "balance": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="balance", provider=PROVIDER)),
    "cash": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="cash", provider=PROVIDER)),
    "ratios": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="ratios", provider=PROVIDER)),
}

{'income':                    accepted_date       revenue  cost_of_revenue  gross_profit  research_and_development_expenses  general_and_administrative_expenses  \
 filing_date                                                                                                                                             
 2025-05-02   2025-05-02 06:00:46   95359000000      50492000000   44867000000                         8550000000                                    0   
 2025-08-01   2025-08-01 06:00:42   94036000000      50318000000   43718000000                         8866000000                                    0   
 2025-10-31   2025-10-31 06:01:26  102466000000      54125000000   48341000000                         8866000000                                    0   
 2026-01-30   2026-01-30 06:01:32  143756000000      74525000000   69231000000                        10887000000                           2095000000   
 2026-05-01   2026-05-01 10:01:00  111184000000      56403000000  

## Events, Estimates, Ownership, and Calendars

In [8]:
if RUN_REFRESH:
    warehouse.refresh_fundamentals(
        EQUITY_SYMBOL,
        sections=["dividends", "historical_splits", "filings", "estimates_price_target", "ownership_insider_trading"],
        providers=[PROVIDER],
    )

{
    "dividends": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="dividends", provider=PROVIDER)),
    "historical_splits": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="historical_splits", provider=PROVIDER)),
    "filings": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="filings", provider=PROVIDER)),
    "estimates_price_target": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="estimates_price_target", provider=PROVIDER)),
}

{'dividends':             record_date  payment_date  declaration_date  adj_dividend  dividend     yield  frequency  amount  adjusted_amount  dividend_yield
 2025-05-12          NaN           NaN               NaN          0.26      0.26  0.479150        NaN     NaN              NaN             NaN
 2025-08-11          NaN           NaN               NaN          0.26      0.26  0.448983        NaN     NaN              NaN             NaN
 2025-11-10          NaN           NaN               NaN          0.26      0.26  0.382289        NaN     NaN              NaN             NaN
 2026-02-09          NaN           NaN               NaN          0.26      0.26  0.378705        NaN     NaN              NaN             NaN
 2026-05-11          NaN           NaN               NaN           NaN       NaN       NaN        NaN    0.27             0.27        0.003588,
 'historical_splits':             numerator  denominator  split_type
 date                                          
 1987-06-16

In [9]:
calendar_routes = {section: fundamental_library(section) for section in EQUITY_CALENDAR_SECTIONS}
route_table(calendar_routes)

,section,openbb_route,provider_library
0,equity_calendar_earnings,equity.calendar.earnings,fmp_equity_fundamental_equity_calendar_earnings
1,equity_calendar_dividend,equity.calendar.dividend,fmp_equity_fundamental_equity_calendar_dividend
2,equity_calendar_splits,equity.calendar.splits,fmp_equity_fundamental_equity_calendar_splits
3,equity_calendar_ipo,equity.calendar.ipo,fmp_equity_fundamental_equity_calendar_ipo


In [10]:
if RUN_REFRESH:
    warehouse.equity_calendar.refresh_all(provider=PROVIDER, start_date="2024-01-01")

state_table("EQUITY_CALENDAR", EQUITY_CALENDAR_SECTIONS)

,symbol,section,provider,min_date,max_date,row_count,column_count,columns_present,last_fetched_at
0,EQUITY_CALENDAR,equity_calendar_earnings,fmp,2005-01-01,2026-06-22,6647,6,"(eps_actual, eps_consensus, last_updated, reve...",2026-06-22T05:36:56.455344+00:00
1,EQUITY_CALENDAR,equity_calendar_dividend,fmp,2005-01-03,2026-06-23,6333,8,"(adjusted_amount, amount, declaration_date, di...",2026-06-22T05:36:57.250837+00:00
2,EQUITY_CALENDAR,equity_calendar_splits,fmp,2005-01-03,2026-06-22,6004,4,"(denominator, numerator, splitType, symbol)",2026-06-22T05:36:57.770534+00:00
3,EQUITY_CALENDAR,equity_calendar_ipo,fmp,2005-01-03,2026-06-22,4772,8,"(actions, exchange, exchange_date, market_cap,...",2026-06-22T05:36:58.161431+00:00


<!-- quant-trader-eda -->
## Quant Trader EDA

Quick checks for a single equity: price risk, drawdowns, benchmark beta, and whether fundamental fields are populated enough to support factor work.

In [11]:
def price_eda(frame: pd.DataFrame, annualization: int = 252) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame is None or frame.empty or "close" not in frame.columns:
        return pd.DataFrame(), pd.DataFrame()
    close = pd.to_numeric(frame["close"], errors="coerce").dropna()
    returns = close.pct_change().dropna()
    if returns.empty:
        return pd.DataFrame(), pd.DataFrame()
    equity_curve = (1 + returns).cumprod()
    drawdown = equity_curve / equity_curve.cummax() - 1
    summary = pd.DataFrame(
        {
            "observations": [int(len(returns))],
            "start": [close.index.min()],
            "end": [close.index.max()],
            "total_return": [close.iloc[-1] / close.iloc[0] - 1],
            "annualized_return": [(1 + returns.mean()) ** annualization - 1],
            "annualized_volatility": [returns.std() * annualization ** 0.5],
            "sharpe_0_rf": [returns.mean() / returns.std() * annualization ** 0.5 if returns.std() else None],
            "max_drawdown": [drawdown.min()],
            "best_day": [returns.max()],
            "worst_day": [returns.min()],
        }
    )
    diagnostics = pd.DataFrame(
        {
            "close": close,
            "return": returns.reindex(close.index),
            "rolling_21d_vol": returns.rolling(21).std() * annualization ** 0.5,
            "rolling_63d_vol": returns.rolling(63).std() * annualization ** 0.5,
            "drawdown": drawdown.reindex(close.index),
        }
    )
    return summary, diagnostics

In [12]:
equity_prices = warehouse.read_prices(EQUITY_SYMBOL, provider=PROVIDER)
equity_summary, equity_diagnostics = price_eda(equity_prices)
equity_summary

,observations,start,end,total_return,annualized_return,annualized_volatility,sharpe_0_rf,max_drawdown,best_day,worst_day
0,11473,1980-12-12,2026-06-24,2982.996603,0.313762,0.43741,0.624227,-0.818139,0.332206,-0.518692


In [13]:
preview(equity_diagnostics[["close", "rolling_21d_vol", "rolling_63d_vol", "drawdown"]], rows=10)

,close,rolling_21d_vol,rolling_63d_vol,drawdown
date,,,,
2026-06-10,291.58,0.217216,0.232647,-0.077979
2026-06-11,295.63,0.221226,0.229781,-0.065172
2026-06-12,291.13,0.221629,0.227224,-0.079402
2026-06-15,296.42,0.231435,0.228787,-0.062674
2026-06-16,299.24,0.232664,0.229127,-0.053757
2026-06-17,295.95,0.234196,0.227374,-0.064160
2026-06-18,298.01,0.235131,0.227107,-0.057646
2026-06-22,297.01,0.231886,0.227043,-0.060808
2026-06-23,294.30,0.230763,0.227167,-0.069378


In [14]:
benchmark = warehouse.etf.read_prices("SPY", provider=PROVIDER)
if equity_prices.empty or benchmark.empty:
    beta_table = pd.DataFrame()
else:
    asset_returns = equity_prices["close"].pct_change().rename(EQUITY_SYMBOL)
    benchmark_returns = benchmark["close"].pct_change().rename("SPY")
    joined = pd.concat([asset_returns, benchmark_returns], axis=1).dropna()
    if joined.empty or joined["SPY"].var() == 0:
        beta_table = pd.DataFrame()
    else:
        beta = joined[EQUITY_SYMBOL].cov(joined["SPY"]) / joined["SPY"].var()
        beta_table = pd.DataFrame({
            "correlation_to_spy": [joined[EQUITY_SYMBOL].corr(joined["SPY"])],
            "beta_to_spy": [beta],
            "tracking_volatility": [(joined[EQUITY_SYMBOL] - joined["SPY"]).std() * 252 ** 0.5],
        })
beta_table

,correlation_to_spy,beta_to_spy,tracking_volatility
0,0.48816,1.106233,0.367953


In [15]:
fundamental_sections = ["income", "balance", "cash", "metrics", "ratios"]
coverage = state_table(EQUITY_SYMBOL, fundamental_sections)
coverage[[column for column in ["section", "row_count", "column_count", "min_date", "max_date"] if column in coverage.columns]]

,section,row_count,column_count,min_date,max_date
0,income,162,32,1985-09-30,2026-05-01
1,balance,150,54,1985-09-30,2026-05-01
2,cash,145,40,1989-12-31,2026-05-01
3,metrics,165,46,1985-09-30,2026-06-25
4,ratios,165,71,1985-09-30,2026-06-25


In [16]:
ratios = warehouse.read_fundamentals(EQUITY_SYMBOL, section="ratios", provider=PROVIDER)
if ratios.empty:
    pd.DataFrame()
else:
    candidate_columns = [
        "price_to_earnings_ratio", "price_to_book_ratio", "price_to_sales_ratio",
        "debt_to_equity_ratio", "return_on_equity", "net_profit_margin",
    ]
    ratios[[column for column in candidate_columns if column in ratios.columns]].tail(8)

<!-- quant-trader-signal-eda -->
## Signal Research for P&L

These cells frame the data as candidate trading signals. They are diagnostics for research, not a recommendation to trade the example symbol.

In [17]:
def signal_backtest(close: pd.Series, signal: pd.Series, annualization: int = 252) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce").dropna()
    returns = close.pct_change().dropna()
    aligned_signal = signal.reindex(returns.index).shift(1).fillna(0).clip(-1, 1)
    strategy_returns = aligned_signal * returns
    if strategy_returns.empty:
        return pd.DataFrame()
    curve = (1 + strategy_returns).cumprod()
    buy_hold = (1 + returns).cumprod()
    drawdown = curve / curve.cummax() - 1
    turnover = aligned_signal.diff().abs().fillna(aligned_signal.abs())
    return pd.DataFrame({
        "strategy_total_return": [curve.iloc[-1] - 1],
        "buy_hold_total_return": [buy_hold.iloc[-1] - 1],
        "strategy_annualized_return": [(1 + strategy_returns.mean()) ** annualization - 1],
        "strategy_annualized_volatility": [strategy_returns.std() * annualization ** 0.5],
        "strategy_sharpe_0_rf": [strategy_returns.mean() / strategy_returns.std() * annualization ** 0.5 if strategy_returns.std() else None],
        "strategy_max_drawdown": [drawdown.min()],
        "hit_rate": [(strategy_returns > 0).mean()],
        "average_exposure": [aligned_signal.abs().mean()],
        "average_daily_turnover": [turnover.mean()],
        "latest_signal": [aligned_signal.iloc[-1]],
    })

In [18]:
equity_prices = warehouse.read_prices(EQUITY_SYMBOL, provider=PROVIDER)
if equity_prices.empty or "close" not in equity_prices.columns:
    pd.DataFrame()
else:
    close = equity_prices["close"]
    trend_signal = (close > close.rolling(200).mean()).astype(float)
    signal_backtest(close, trend_signal)

In [19]:
if equity_prices.empty or "close" not in equity_prices.columns:
    pd.DataFrame()
else:
    close = equity_prices["close"]
    returns = close.pct_change()
    vol_21 = returns.rolling(21).std() * 252 ** 0.5
    trend = close / close.rolling(200).mean() - 1
    breakout = close / close.rolling(63).max() - 1
    candidate = pd.DataFrame({
        "close": close,
        "trend_200d_distance": trend,
        "breakout_63d_distance": breakout,
        "realized_vol_21d": vol_21,
        "vol_target_weight_15pct_cap_1x": (0.15 / vol_21).clip(0, 1),
        "risk_state": pd.cut(vol_21, bins=[-float("inf"), 0.2, 0.4, float("inf")], labels=["normal", "elevated", "high"]),
    })
    preview(candidate, rows=15)

In [20]:
ratios = warehouse.read_fundamentals(EQUITY_SYMBOL, section="ratios", provider=PROVIDER)
metrics = warehouse.read_fundamentals(EQUITY_SYMBOL, section="metrics", provider=PROVIDER)
frames = []
for name, frame in {"ratios": ratios, "metrics": metrics}.items():
    if not frame.empty:
        latest = frame.tail(1).copy()
        latest.index = [name]
        frames.append(latest)
if not frames:
    pd.DataFrame()
else:
    latest_fundamentals = pd.concat(frames, axis=0)
    signal_columns = [
        "price_to_earnings_ratio", "price_to_book_ratio", "price_to_sales_ratio",
        "return_on_equity", "net_profit_margin", "debt_to_equity_ratio",
        "free_cash_flow_yield", "earnings_yield",
    ]
    latest_fundamentals[[column for column in signal_columns if column in latest_fundamentals.columns]]

<!-- output-analysis -->
## Analysis Notes From This Run

For `AAPL`, the stored FMP history is strongly up over the full sample but has also experienced an 81.8% max drawdown, so raw long-only exposure has meaningful crash risk. The latest stored close is above the 200-day average by about 9.1%, and the 21-day realized volatility is about 22.5%, which is a constructive trend state but not a low-risk one.

Against `SPY`, the daily correlation is about 0.49 and the estimated beta/tracking behavior is materially idiosyncratic. For a quant trader, this makes `AAPL` more useful as a single-name alpha/risk sleeve than as a market proxy. The practical next checks are whether valuation/fundamental signals improve entries and whether volatility targeting reduces the large historical drawdown.